In [1]:
import pandas as pd

df = pd.read_csv("labeled_table.csv")

print("Shape:", df.shape)

print("\nLabel distribution:")
print(df["is_late"].value_counts())

print("\nLabel percentages:")
print(df["is_late"].value_counts(normalize=True) * 100)

Shape: (96476, 24)

Label distribution:
is_late
0    88649
1     7827
Name: count, dtype: int64

Label percentages:
is_late
0    91.887101
1     8.112899
Name: proportion, dtype: float64


In [2]:
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"],
    errors="coerce"
)

print("Start date:", df["order_purchase_timestamp"].min())
print("End date:", df["order_purchase_timestamp"].max())

print("\nOrders by year:")
print(df["order_purchase_timestamp"].dt.year.value_counts().sort_index())

print("\nLate rate by year:")
print(
    df.groupby(df["order_purchase_timestamp"].dt.year)["is_late"]
      .mean()
      .mul(100)
)

Start date: 2016-09-15 12:16:38
End date: 2018-08-29 15:00:37

Orders by year:
order_purchase_timestamp
2016      272
2017    43426
2018    52778
Name: count, dtype: int64

Late rate by year:
order_purchase_timestamp
2016    1.470588
2017    6.627366
2018    9.369434
Name: is_late, dtype: float64


In [3]:
# Sort data chronologically
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

# Time-based split: 70% Train, 15% Validation, 15% Test
n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain dates:")
print(train_df["order_purchase_timestamp"].min(), "->",
      train_df["order_purchase_timestamp"].max())

print("\nValidation dates:")
print(val_df["order_purchase_timestamp"].min(), "->",
      val_df["order_purchase_timestamp"].max())

print("\nTest dates:")
print(test_df["order_purchase_timestamp"].min(), "->",
      test_df["order_purchase_timestamp"].max())

Train shape: (67533, 24)
Validation shape: (14471, 24)
Test shape: (14472, 24)

Train dates:
2016-09-15 12:16:38 -> 2018-04-15 20:07:56

Validation dates:
2018-04-15 20:10:23 -> 2018-06-21 07:50:39

Test dates:
2018-06-21 08:29:29 -> 2018-08-29 15:00:37


In [4]:
for name, split in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    late_rate = split["is_late"].mean() * 100
    
    print(
        f"{name}: "
        f"{len(split)} rows | "
        f"Late = {late_rate:.2f}%"
    )

Train: 67533 rows | Late = 9.03%
Validation: 14471 rows | Late = 5.34%
Test: 14472 rows | Late = 6.61%


In [5]:
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("Train, validation, and test files saved successfully!")

Train, validation, and test files saved successfully!
